# MobileFaceNet (tflite) → CoreML (.mlpackage) — TEK SEFERLİK dönüşüm

Android `mobilefacenet.tflite` (192-dim) → iOS CoreML `.mlpackage`. **1 KEZ** çalışır;
çıktı repoya commit edilir (Xcode her build'de yalnızca *derler*, coremltools'u TEKRAR çalıştırmaz).

## Kullanım (tek hücre)
1. **Runtime → Restart session** (temiz başlangıç).
2. Aşağıdaki **tek kod hücresini** çalıştır (Shift+Enter).
3. Sorulduğunda `mobilefacenet.tflite`'ı yükle
   (`src/VerifyBlind.Android/app/src/main/assets/mobilefacenet.tflite`).
4. `PARITE cosine = 0.99x` görmelisin (dönüşüm sadık).
5. İnen `MobileFaceNet.mlpackage.zip`'i aç → `MobileFaceNet.mlpackage`'ı `Resources/`'a koy + commit.

Notlar: Colab'ın hazır torch/tensorflow'u kullanılır (PİN YOK). NHWC/NCHW layout farkı
otomatik ele alınır. Modeldeki son `LpNormalization` (L2) çıkarılır — uygulama (`FaceEmbedder`)
ve Android zaten L2-normalize ettiği için sonuç birebir aynıdır. Parite **torch** ile yapılır
(coremltools `.predict()` macOS ister; Colab Linux). Sözleşme: 112×112 RGB image girdi,
`(p-127.5)/128` gömülü, 192-dim çıktı.

In [ ]:
# ===== MobileFaceNet tflite → CoreML — TEK HÜCRE =====
# 1) Eksik kütüphaneler (Colab'ın torch/tensorflow'unu kullan, PİN YOK)
!pip install -q coremltools onnx onnx2torch tf2onnx

import sys, subprocess, numpy as np, torch, onnx, tensorflow as tf, coremltools as ct
from onnx2torch import convert
from google.colab import files

# 2) tflite yükle
print('>>> mobilefacenet.tflite dosyasini secin:')
up = files.upload()
TFLITE = next(iter(up.keys()))
print('Yuklendi:', TFLITE)

# 3) tflite -> onnx
subprocess.run([sys.executable, '-m', 'tf2onnx.convert', '--tflite', TFLITE,
                '--output', 'mobilefacenet.onnx', '--opset', '13'], check=True)

# 4) LpNormalization'i cikar (app zaten L2-normalize eder) + onnx -> torch (layout-robust)
raw = onnx.load('mobilefacenet.onnx')
for n in [x for x in raw.graph.node if x.op_type == 'LpNormalization']:
    for o in raw.graph.output:
        if o.name == n.output[0]: o.name = n.input[0]
    raw.graph.node.remove(n)
onnx.save(raw, 'mobilefacenet_noLp.onnx')
onnx_model = onnx.load('mobilefacenet_noLp.onnx')
inner = convert(onnx_model).eval()
shape = [d.dim_value for d in onnx_model.graph.input[0].type.tensor_type.shape.dim]
print('onnx input shape:', shape)
channels_last = (len(shape) == 4 and shape[-1] == 3)  # NHWC mi?

class Wrap(torch.nn.Module):
    def __init__(self, m, cl):
        super().__init__(); self.m = m; self.cl = cl
    def forward(self, x):                      # x: NCHW (1,3,112,112)
        if self.cl: x = x.permute(0, 2, 3, 1).contiguous()
        return self.m(x)

model = Wrap(inner, channels_last).eval()
example = torch.rand(1, 3, 112, 112)
print('torch cikti boyutu:', tuple(model(example).shape))
traced = torch.jit.trace(model, example)

# 5) torch -> CoreML (112x112 RGB image, normalizasyon gomulu)
mlmodel = ct.convert(
    traced,
    inputs=[ct.ImageType(name='image', shape=(1, 3, 112, 112),
                         scale=1/128.0, bias=[-127.5/128.0]*3,
                         color_layout=ct.colorlayout.RGB)],
    minimum_deployment_target=ct.target.iOS16,
    convert_to='mlprogram',
)
mlmodel.short_description = 'MobileFaceNet 112x112 -> 192-dim (Android parity).'
mlmodel.save('MobileFaceNet.mlpackage')
print('CoreML kaydedildi.')

# 6) PARITE: tflite vs torch (Linux'ta calisir; coreml.predict macOS ister)
#    torch model = tflite eksi LpNorm; ikisini de L2-normalize edip karsilastiririz.
rng = np.random.default_rng(42)
img = rng.integers(0, 256, (112, 112, 3), dtype=np.uint8)
interp = tf.lite.Interpreter(model_path=TFLITE); interp.allocate_tensors()
ind, outd = interp.get_input_details()[0], interp.get_output_details()[0]
x = ((img.astype(np.float32) - 127.5) / 128.0)[None, ...]      # tflite NHWC
if list(ind['shape'])[1] == 3: x = np.transpose(x, (0, 3, 1, 2))
interp.set_tensor(ind['index'], x); interp.invoke()
e1 = interp.get_tensor(outd['index']).reshape(-1)
xt = np.transpose(((img.astype(np.float32) - 127.5) / 128.0), (2, 0, 1))[None, ...]
with torch.no_grad():
    e2 = model(torch.from_numpy(xt)).numpy().reshape(-1)
l2 = lambda v: v / (np.linalg.norm(v) or 1)
print('PARITE cosine (tflite vs torch) =', round(float(np.dot(l2(e1), l2(e2))), 5), '(>=0.99 sadik)')

# 7) zip + indir
import shutil
shutil.make_archive('MobileFaceNet.mlpackage', 'zip', 'MobileFaceNet.mlpackage')
files.download('MobileFaceNet.mlpackage.zip')

## Son adım (repoda)
1. İnen `MobileFaceNet.mlpackage.zip`'i aç.
2. Çıkan `MobileFaceNet.mlpackage` klasörünü `src/VerifyBlind.iOS/Resources/` içine koy.
3. Commit + push (`dev`). Sonraki build'de `FaceEmbedder` modeli bulur → canlı % aktif olur.